# Test Butler from LSSTCam

In [ ]:

import sys
import os
import matplotlib.pyplot as plt
import lsst.afw.display as afwDisplay
import numpy as np
import pandas as pd
from astropy.time import Time

In [ ]:
from lsst.daf.butler import Butler

In [ ]:
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.labelsize"] = "x-large"
plt.rcParams["axes.titlesize"] = "x-large"
plt.rcParams["xtick.labelsize"] = "x-large"
plt.rcParams["ytick.labelsize"] = "x-large"


In [ ]:
!eups list lsst_distrib

In [ ]:
repo = "main"
instrument = "LSSTCam"
date_start = 20250415
date_selection = 20250416
where_clause = "instrument = '" + f"{instrument}" + "'"
where_clause_date = where_clause + f"and day_obs >= {date_start}"
skymapName = "lsst_cells_v1"
COLLECTION_NAME = 'LSSTCam/raw/all' 

In [ ]:
os.getenv("LSST_RESOURCES_WEBDAV_CONFIG")

In [ ]:
os.getenv("DAF_BUTLER_REPOSITORY_INDEX")

## Butler init

In [ ]:
#!eups list -s

In [ ]:
butler = Butler(repo)
registry = butler.registry

In [ ]:
butler.registry.queryDatasetTypes()

In [ ]:
all_collections = registry.queryCollections()

print(f"Nombre total de collections trouvées : {len(all_collections)}")

# Afficher les 20 premières pour un aperçu
print("\nLes 20 premières collections :")
for i, collection_name in enumerate(all_collections):
    print(collection_name)
    if i >= 19:
        break

In [ ]:
# Essayez la même méthode, mais sur l'objet 'registry'
try:
    exposure_records = registry.queryDimensionRecords('exposure')
    
    # Le reste de votre code
    list_of_records = list(exposure_records)
    print(f"Nombre total d'expositions trouvées : {len(list_of_records)}")

except AttributeError as e:
    print(f"Erreur de l'API Gen 3 : {e}")
    print("Veuillez vérifier l'initialisation de votre Butler, notamment l'URI et les collections.")

In [ ]:
df_exposure = pd.DataFrame(
    {
        "id": pd.Series(dtype="int"),
        "obs_id": pd.Series(dtype="int"),
        "day_obs": pd.Series(dtype="int"),
        "seq_num": pd.Series(dtype="int"),
        "time_start": pd.Series(dtype="str"),  # ou 'datetime64[ns]' si c’est un datetime
        "time_end": pd.Series(dtype="str"),  # idem
        "type": pd.Series(dtype="str"),
        "target": pd.Series(dtype="str"),
        "filter": pd.Series(dtype="str"),
        "zenith_angle": pd.Series(dtype="float"),
        "expos": pd.Series(dtype="float"),  # ou 'int' selon le cas
        "ra": pd.Series(dtype="float"),
        "dec": pd.Series(dtype="float"),
        "skyangle": pd.Series(dtype="float"),
        "azimuth": pd.Series(dtype="float"),
        "zenith": pd.Series(dtype="float"),
        "science_program": pd.Series(dtype="str"),
        "jd": pd.Series(dtype="float"),
        "mjd": pd.Series(dtype="float"),
    }
)

In [ ]:
# save the data array in rows before saving in pandas dataframe
rows = []
for count, info in enumerate(registry.queryDimensionRecords("exposure", where=where_clause_date)):
    try:
        jd_start = info.timespan.begin.value
        jd_end = info.timespan.end.value
        the_Time_start = Time(jd_start, format="jd", scale="utc")
        the_Time_end = Time(jd_end, format="jd", scale="utc")
        mjd_start = the_Time_start.mjd
        mjd_end = the_Time_end.mjd
        isot_start = the_Time_start.isot
        isot_end = the_Time_end.isot

        if count == 0:
            print("===== Time Conversion Debug Info =====")
            print(f"JD start      : {jd_start} (type: {type(jd_start)})")
            print(f"JD end        : {jd_end} (type: {type(jd_end)})")
            print(f"MJD start     : {mjd_start} (type: {type(mjd_start)})")
            print(f"MJD end       : {mjd_end} (type: {type(mjd_end)})")
            print(f"ISOT start    : {isot_start} (type: {type(isot_start)})")
            print(f"ISOT end      : {isot_end} (type: {type(isot_end)})")
            print("=======================================")

        # put row in a dictionnary before stacking
        row = {
            "id": info.id,
            "obs_id": info.obs_id,
            "day_obs": info.day_obs,
            "seq_num": info.seq_num,
            "time_start": isot_start,
            "time_end": isot_end,
            "type": info.observation_type,
            "target": info.target_name,
            "filter": info.physical_filter,
            "zenith_angle": info.zenith_angle,
            "expos": info.exposure_time,  # Exemple : adapter selon ton objet
            "ra": info.tracking_ra,
            "dec": info.tracking_dec,
            "skyangle": info.sky_angle,
            "azimuth": info.azimuth,
            "zenith": info.zenith_angle,
            "science_program": info.science_program,
            "jd": float(jd_start),
            "mjd": float(mjd_start),
        }
        rows.append(row)

    except ValueError as e:
        print(f"Erreur de valeur : {e}")
    except FileNotFoundError as e:
        print(f"Fichier introuvable : {e}")
    except Exception as e:
        print(f"Erreur inattendue : {type(e).__name__} - {e}")
        print(f">>>   Unexpected error at row {count}:", sys.exc_info()[0])
        traceback.print_exc()  # affiche la stack trace complète


In [ ]:
# Création finale du DataFrame
df_exposure = pd.DataFrame(rows)

In [ ]:
df_exposure

In [ ]:
df_exposure["target"].unique()

In [ ]:
df_science = df_exposure[df_exposure.type == "science"]
df_science.reset_index(drop=True, inplace=True)

In [ ]:
df_science = df_science.copy()
df_science["band"] = df_science["filter"].apply(lambda x : x.split("_")[0])

In [ ]:
df_science